# OptiGuard Step 8: Neural Restoration Model Training
### Constrained-Gain Model Selection on Synthetic Raman Hyperspectral Maps

This notebook runs the complete Step 8 pipeline:
1. **Environment Setup & Public Repo Clone**
2. **Physics Gate Verification (Merged single pytest run)**
3. **Synthetic Corpus Generation (300 Datacubes)**
4. **Spatial-Spectral U-Net Training with Validation Every 5 Epochs**
5. **Harness Evaluation & Baseline Comparison**
6. **ONNX Export & Inference Benchmark**

In [ ]:
# Cell 1: Environment Setup & Clone
import os

GH_USER = 'yashwanth-maram'
REPO = 'Optiguard'
CLONE_URL = f'https://github.com/{GH_USER}/{REPO}.git'

print(f"Cloning {CLONE_URL}...")
!rm -rf Optiguard
!git clone {CLONE_URL}
%cd Optiguard

!pip install -q -e ".[train,serve,dev]"
print("Installation complete.")

In [ ]:
# Cell 2: Physics Gate & Baseline Sanity Check (Merged single run)
!pytest tests/test_physics.py tests/test_baselines.py -k "not slow" -v

In [ ]:
# Cell 3: Generate 300 Synthetic Raman Datacubes
!python scripts/generate_corpus.py --seed 20260806 --n 300 --window 128 --out /content/data/corpus

In [ ]:
# Cell 4: Train Spatial-Spectral Restoration Network (Validating every 5 epochs)
!python training/train.py \
    --config configs/restoration_v1.yaml \
    --data /content/data/corpus \
    --out /content/runs/restoration_v1 \
    --select-on constrained_gain \
    --recall-floor 0.65 \
    --checkpoint-every 5

In [ ]:
# Cell 5: Evaluate Best Restoration Model on Test Maps
import json
import numpy as np
from optiguard.data.simulator import MapSimulator
from optiguard.eval.harness import evaluate
from optiguard.models.wrapper import load_restoration_method

sim = MapSimulator.from_yaml("configs/simulator.yaml")
test_samples = [sim.generate(index=i) for i in range(6, 12)]

best_checkpoint = "/content/runs/restoration_v1/best.pt"
restoration_fn = load_restoration_method(best_checkpoint, config_path="configs/restoration_v1.yaml")

res = evaluate(restoration_fn, test_samples, exposure=0.1)

print("=" * 60)
print("OPTIGUARD RESTORATION V1 EVALUATION RESULTS (0.1s exposure)")
print("=" * 60)
print(f"Effective Exposure Gain:  {res.effective_exposure_gain:.2f}x")
print(f"RMSE Center Error:       {res.rmse_center_cm1:.4f} cm^-1")
print(f"MAE Center Error:        {res.mae_center_cm1:.4f} cm^-1")
print(f"False Feature Rate:      {res.false_feature_rate*100:.2f}%")
print("\nDefect Recall by Difficulty (CRLB multiples):")
for diff, rec in sorted(res.recall_by_difficulty.items()):
    print(f"  {diff:4.1f} CRLB: {rec*100:5.1f}%")
print("=" * 60)

In [ ]:
# Cell 6: Export to ONNX & Run Benchmark
!python scripts/export_onnx.py \
    --checkpoint /content/runs/restoration_v1/best.pt \
    --out /content/runs/restoration_v1/model.onnx \
    --config configs/restoration_v1.yaml

# Optional download link in Colab
from google.colab import files
files.download('/content/runs/restoration_v1/best.pt')
files.download('/content/runs/restoration_v1/model.onnx')